# Advanced Python Custom Exceptions — A Step-by-Step Problem-Solving Tutorial

**18 entirely new advanced problems · full worked solutions · incremental code · extensive explanations · executable assertions**

This notebook follows the teaching style of the supplied *Custom Exceptions* tutorial: explain the motivation, construct the exception in small pieces, observe what Python does, refine the design, and verify it with code. It does **not** copy the previous 20-problem notebook or simply present one large solution per exercise.

**Runtime:** Python 3.11 or later (for `add_note`, `ExceptionGroup`, `except*`, and `asyncio.TaskGroup`). **Dependencies:** standard library only. All examples are simulated or run in memory; no network, credentials, or external database needed.

**Recommended study method:** read a problem, predict each code cell’s output, run it, inspect the explanation, then alter the test input and re-run. Run all cells top-to-bottom in a fresh kernel. Names are prefixed or deliberately localized to make the sequential run reproducible.

> Production guideline: catch the narrowest error you can handle; do not silently turn arbitrary programming mistakes into business failures. Avoid logging secrets or returning implementation details to end users.

## Road map

| Problems | Theme |
|---|---|
| 01–04 | Exception contracts, field invariants, chaining, and diagnostic notes |
| 05–09 | Decorators, hierarchical validation, context managers, resource acquisition, cleanup failures |
| 10–13 | Generator boundaries, cache safety, batch policies, circuit breakers |
| 14–16 | Task groups, asynchronous timeouts, and SQLite transaction boundaries |
| 17–18 | Safe inheritance composition and an integrated import-pipeline capstone |

The introductory source notebook shows subclassing `Exception`, building a hierarchy, attaching public HTTP-facing information, and multiple inheritance. Here we retain its explain → code → explain rhythm while examining *new* failure modes and safer production patterns. In particular, **multiple inheritance of distinct built-in exception types is not generally recommended**: their C-level layouts and initialization semantics may conflict. We instead demonstrate composition with a normal mixin in Problem 17.

## Shared setup: deterministic teaching helpers

Only standard-library imports are used. `capture_exception` deliberately catches `Exception` **for a test demonstration**; this is not permission to write blanket exception handlers in application code. It returns the error so we can inspect it without generating an uncaught traceback in the notebook.

In [1]:
from __future__ import annotations

import asyncio
import functools
import io
import json
import sqlite3
import traceback
from contextlib import ExitStack, contextmanager
from dataclasses import dataclass
from enum import Enum
from typing import Callable


def capture_exception(operation, *args, **kwargs):
    """Test-only: return a raised Exception or fail if no exception occurs."""
    try:
        operation(*args, **kwargs)
    except Exception as exc:
        return exc
    raise AssertionError("Expected an exception, but operation succeeded")


print("Setup complete; standard library only.")

Setup complete; standard library only.


## Problem 01 — Design a public contract that survives a changing hierarchy

**Scenario / challenge.** A workflow library has missing-step and duplicate-step errors. Clients should be able to catch one common application error, and still inspect a specific failure.

**You will be able to:** design a root class; order handlers correctly; prove which exceptions do and do not belong to the library.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Start with the caller’s needs

A root exception lets a caller say “recover from any *documented workflow failure*” without `except Exception`, which would also swallow unrelated mistakes. Specialized subclasses provide finer-grained recovery. A programmer’s `TypeError` should **not** automatically become a workflow error.

In [2]:
class P01WorkflowError(Exception):
    """Base class for documented workflow failures."""


class P01MissingStep(P01WorkflowError):
    """A requested step does not exist."""


class P01DuplicateStep(P01WorkflowError):
    """A step name has already been registered."""


print(issubclass(P01MissingStep, P01WorkflowError))
print(issubclass(TypeError, P01WorkflowError))

True
False


The first result is `True`: a missing step can be caught through the shared root. The second result is `False`: ordinary Python programming errors stay outside our public library taxonomy.

### Step 2 — Write the operation before the error handler

The library enforces simple invariants: names must be strings, duplicates are forbidden, and missing keys are domain failures. Built-in `TypeError` remains appropriate for a wrong *type* supplied by a caller.

In [3]:
class P01Registry:
    """Small registry with explicit exception guarantees."""

    def __init__(self):
        self._steps = {}

    def add(self, name: str, operation: Callable):
        if not isinstance(name, str):
            raise TypeError("name must be a string")
        if name in self._steps:
            raise P01DuplicateStep(f"step already exists: {name}")
        self._steps[name] = operation

    def get(self, name: str):
        try:
            return self._steps[name]
        except KeyError as exc:
            raise P01MissingStep(f"unknown step: {name}") from exc


p01_registry = P01Registry()
p01_registry.add("clean", str.strip)
print(p01_registry.get("clean")("  example  "))

example


Notice the narrow `except KeyError` and explicit `raise ... from exc`. We translate a storage detail to a meaningful public exception at a boundary; we do not convert all unexpected errors.

### Step 3 — Demonstrate handler ordering

Python checks handlers in source order. A generic root handler placed first would intercept every subclass. Place specific handlers first, and the shared root later.

In [4]:
def p01_describe(operation):
    try:
        operation()
    except P01MissingStep:
        return "missing: caller may choose an alternative"
    except P01WorkflowError:
        return "other documented workflow error"


print(p01_describe(lambda: p01_registry.get("unknown")))
print(p01_describe(lambda: p01_registry.add("clean", str.strip)))

missing: caller may choose an alternative
other documented workflow error


### Verify the solution

Exercise both leaf exceptions and verify that wrong-type programmer errors remain outside the hierarchy.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [5]:
assert p01_describe(lambda: p01_registry.get("unknown")) == "missing: caller may choose an alternative"
assert p01_describe(lambda: p01_registry.add("clean", str.strip)) == "other documented workflow error"
assert isinstance(capture_exception(p01_registry.add, 123, str.strip), TypeError)
assert not isinstance(capture_exception(p01_registry.add, 123, str.strip), P01WorkflowError)
print("PASS 01 — hierarchy and catch order")

PASS 01 — hierarchy and catch order


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 02 — Preserve args while validating structured error fields

**Scenario / challenge.** A storage API needs an exception carrying a numeric shard ID, a safe public message, and a machine-readable code. An invalid shard must fail during exception construction.

**You will be able to:** define an exception constructor; keep `Exception.args` meaningful; enforce metadata invariants.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Know what Exception already gives you

The base class stores positional arguments in `args`, and `str(exception)` usually derives from those arguments. An override should call `super().__init__(message)` unless a very specific compatibility contract says otherwise.

In [6]:
plain_error = Exception("disk temporarily unreachable")
print(plain_error.args)
print(str(plain_error))
print(repr(plain_error))

('disk temporarily unreachable',)
disk temporarily unreachable
Exception('disk temporarily unreachable')


A useful custom class should preserve this familiar behavior. Downstream logging, tracing, and assertion tools often depend on `args` and `str(error)` being coherent.

### Step 2 — Separate the human message from structured data

Do not squeeze all fields into an arbitrary tuple and require callers to decode positions. Store named fields, validate their types, and pass a single diagnostic message to `Exception`.

In [7]:
class P02StorageError(Exception):
    """Storage failure with validated, inspectable metadata."""

    code = "STORAGE_ERROR"
    public_message = "Storage is temporarily unavailable."

    def __init__(self, message: str, *, shard: int):
        if not isinstance(message, str) or not message:
            raise ValueError("message must be a non-empty string")
        if isinstance(shard, bool) or not isinstance(shard, int) or shard < 0:
            raise ValueError("shard must be a non-negative integer")
        self.shard = shard
        super().__init__(message)


failure = P02StorageError("replica is behind", shard=7)
print(failure.args, failure.shard, failure.code)
print(str(failure), "|", failure.public_message)

('replica is behind',) 7 STORAGE_ERROR
replica is behind | Storage is temporarily unavailable.


The public message is a safe class-level default; the internal message can provide operational context. A boolean is deliberately rejected as a shard even though `isinstance(True, int)` is `True` in Python.

### Step 3 — Make it subclassable without surprising callers

The constructor owns validated fields. A more specific leaf can inherit its behavior and override only constant metadata; no duplicated `__init__` is necessary.

In [8]:
class P02ReplicaLag(P02StorageError):
    """Replica has not caught up with the write log."""

    code = "REPLICA_LAG"


p02_error = P02ReplicaLag("lag exceeds 30 seconds", shard=2)
print(type(p02_error).__name__, p02_error.args, p02_error.shard)

P02ReplicaLag ('lag exceeds 30 seconds',) 2


### Verify the solution

Check message fidelity, immutable code inheritance, and rejection of invalid shard values.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [9]:
assert p02_error.args == ("lag exceeds 30 seconds",)
assert p02_error.shard == 2
assert p02_error.code == "REPLICA_LAG"
for invalid in (-1, True, "2", 2.5):
    assert isinstance(capture_exception(P02StorageError, "bad", shard=invalid), ValueError)
assert isinstance(capture_exception(P02StorageError, "", shard=0), ValueError)
print("PASS 02 — validated exception payload")

PASS 02 — validated exception payload


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 03 — Inspect implicit context, explicit cause, and suppressed display

**Scenario / challenge.** A configuration loader wraps an invalid integer conversion. You need to explain three different traceback behaviors without discarding debugging information.

**You will be able to:** distinguish `__context__`, `__cause__`, and `__suppress_context__`; choose clear translation syntax.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Observe automatic exception context

When an exception is raised while another exception is being handled, Python records the earlier exception as `__context__`, even without `from`. This records how the later failure arose.

In [10]:
def p03_implicit():
    try:
        int("not-an-integer")
    except ValueError:
        raise P01WorkflowError("configuration could not be loaded")


p03_a = capture_exception(p03_implicit)
print(type(p03_a.__context__).__name__, p03_a.__cause__)
print(p03_a.__suppress_context__)

ValueError None
False


Here the low-level `ValueError` is available through `__context__`, but `__cause__` is `None`. This is an *implicit* chain. The context explains where the workflow failure came from.

### Step 2 — Use explicit cause for deliberate translation

When you intentionally replace a dependency exception with a domain exception, `raise DomainError(...) from exc` makes the relationship unambiguous and sets the explicit cause.

In [11]:
def p03_explicit():
    try:
        int("not-an-integer")
    except ValueError as exc:
        raise P01WorkflowError("configuration could not be loaded") from exc


p03_b = capture_exception(p03_explicit)
print(type(p03_b.__cause__).__name__)
print(p03_b.__suppress_context__)

ValueError
True


The cause now points directly to `ValueError`; Python displays it as the direct cause in a standard traceback. This is the usual approach for intentional error translation at a public boundary.

### Step 3 — Understand what `from None` actually hides

Use `raise ... from None` only when an underlying exception is irrelevant to *display*. It does **not** remove the low-level object: Python still stores the context, so it is not a security or redaction mechanism.

In [12]:
def p03_suppressed():
    try:
        int("not-an-integer")
    except ValueError:
        raise P01WorkflowError("invalid configuration") from None


p03_c = capture_exception(p03_suppressed)
rendered = "".join(traceback.format_exception(p03_c))
print("stored context:", type(p03_c.__context__).__name__)
print("context displayed:", "invalid literal" in rendered)
print("suppression flag:", p03_c.__suppress_context__)

stored context: ValueError
context displayed: False
suppression flag: True


Because traceback formatting honors suppression, the old error text is not displayed as the active traceback chain. But the object remains reachable. Never embed credentials in exception messages or treat `from None` as redaction.

### Verify the solution

Verify three independent observable contracts instead of comparing version-specific traceback formatting.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [13]:
assert isinstance(p03_a.__context__, ValueError)
assert p03_a.__cause__ is None
assert isinstance(p03_b.__cause__, ValueError)
assert p03_b.__suppress_context__
assert isinstance(p03_c.__context__, ValueError)
assert p03_c.__cause__ is None and p03_c.__suppress_context__
print("PASS 03 — context, cause, and suppression")

PASS 03 — context, cause, and suppression


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 04 — Enrich an exception with PEP 678 diagnostic notes

**Scenario / challenge.** A job goes through a parser, transformer, and writer. Each layer should attach its own diagnostic clue without changing the exception type or overwriting the original message.

**You will be able to:** use `add_note`; retain original exception identity; keep notes free of secrets.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Explore the note mechanism

Python 3.11 added `BaseException.add_note(text)`. Notes are displayed with a traceback and stored in `__notes__`. They complement the error message, rather than replace it.

In [14]:
p04_demo = ValueError("record rejected")
p04_demo.add_note("phase=parsing")
p04_demo.add_note("source=demo.csv")
print(str(p04_demo))
print(p04_demo.__notes__)

record rejected
['phase=parsing', 'source=demo.csv']


The `str()` output remains the original message. A logging system or traceback renderer can show the notes. This makes notes a useful way to add *where* a failure occurred without manufacturing a new exception at every layer.

### Step 2 — Annotate at the right boundary, and re-raise

Use a narrow catch around the operation whose context you know. Bare `raise` re-raises the same object with its original type and traceback. Add only non-sensitive data.

In [15]:
class P04RecordError(Exception):
    """A record cannot be transformed."""


def p04_parse(value):
    if not value.isdigit():
        raise P04RecordError("record ID must contain digits")
    return int(value)


def p04_transform(value, *, filename, row_number):
    try:
        return p04_parse(value) * 10
    except P04RecordError as exc:
        exc.add_note(f"file={filename}")
        exc.add_note(f"row={row_number}")
        raise


p04_error = capture_exception(p04_transform, "bad", filename="sample.csv", row_number=4)
print(type(p04_error).__name__)
print(p04_error.__notes__)

P04RecordError
['file=sample.csv', 'row=4']


There is still one `P04RecordError` rather than nested wrappers. Additional layers could add more notes before re-raising. In production, treat file names and identifiers as potentially sensitive unless safe for your logging audience.

### Step 3 — Show how notes appear in a traceback

Instead of deliberately crashing the notebook, render the captured exception to a string. This preserves the demonstration while keeping all cells executable.

In [16]:
p04_rendered = "".join(traceback.format_exception(p04_error))
print("message present:", "record ID must contain digits" in p04_rendered)
print("file note present:", "file=sample.csv" in p04_rendered)
print("row note present:", "row=4" in p04_rendered)

message present: True
file note present: True
row note present: True


### Verify the solution

Check the original exception type, the unchanged message, and separately attached notes.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [17]:
assert type(p04_error) is P04RecordError
assert str(p04_error) == "record ID must contain digits"
assert p04_error.__notes__ == ["file=sample.csv", "row=4"]
assert p04_transform("12", filename="sample.csv", row_number=1) == 120
print("PASS 04 — composable diagnostic notes")

PASS 04 — composable diagnostic notes


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 05 — Translate dependency failures with a type-preserving decorator

**Scenario / challenge.** A repository method uses an unreliable low-level file adapter. The public method should raise a repository-specific exception for `OSError`, but should preserve `ValueError` programming or validation errors.

**You will be able to:** write a decorator using `functools.wraps`; catch only the intended boundary; preserve the original cause.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Declare the boundary and its public type

An adapter may expose low-level operating-system exceptions. The repository promises a stable custom error for those failures. It should not claim unrelated bugs were I/O failures.

In [18]:
class P05RepositoryError(Exception):
    """A repository operation failed at its I/O boundary."""


def p05_translate_io(function):
    """Translate only OS-level adapter errors."""
    @functools.wraps(function)
    def wrapper(*args, **kwargs):
        try:
            return function(*args, **kwargs)
        except OSError as exc:
            raise P05RepositoryError("repository I/O operation failed") from exc
    return wrapper

The decorated function’s `__name__` and docstring survive thanks to `functools.wraps`. More importantly, the exception handler knows exactly which low-level family it translates.

### Step 2 — Exercise normal, expected-failure, and unexpected-failure paths

A small fake adapter is enough to show the contract. We can deliberately inject each failure without a real filesystem.

In [19]:
@p05_translate_io
def p05_read(mode):
    """Read a fake record from an adapter."""
    if mode == "io":
        raise OSError("adapter disconnected")
    if mode == "bug":
        raise ValueError("invalid adapter state")
    return "record-42"


print(p05_read("ok"))
print(p05_read.__name__, p05_read.__doc__)
p05_io_error = capture_exception(p05_read, "io")
print(type(p05_io_error).__name__, type(p05_io_error.__cause__).__name__)

record-42
p05_read Read a fake record from an adapter.
P05RepositoryError OSError


`ValueError` should propagate unchanged, not be mislabeled. When users report a bug, honest exception types make both diagnostics and recovery more reliable.

### Step 3 — Check that the wrapper does not overreach

We test the identity of the unexpected exception type and the explicit cause of the translated exception. This prevents future refactoring from accidentally expanding the `except` clause.

In [20]:
p05_unexpected = capture_exception(p05_read, "bug")
print(type(p05_unexpected).__name__)
print("translated?", isinstance(p05_unexpected, P05RepositoryError))

ValueError
translated? False


### Verify the solution

Prove success, metadata preservation, precise translation, and non-translation.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [21]:
assert p05_read("ok") == "record-42"
assert p05_read.__name__ == "p05_read"
assert isinstance(p05_io_error, P05RepositoryError)
assert isinstance(p05_io_error.__cause__, OSError)
assert type(p05_unexpected) is ValueError
print("PASS 05 — narrow decorator boundary")

PASS 05 — narrow decorator boundary


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 06 — Validate a nested configuration with precise field paths

**Scenario / challenge.** A service accepts nested Python dictionaries. Rather than return a cryptic `KeyError`, report exactly which field is invalid and aggregate independent issues before raising a validation exception.

**You will be able to:** design issue objects; construct key paths; aggregate and test multiple independent failures.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Model one issue separately from the exception

A validation issue is useful *data*, not necessarily an exception itself. A frozen dataclass describes the path and reason; an exception represents the final decision that the entire configuration cannot be accepted.

In [22]:
@dataclass(frozen=True)
class P06Issue:
    path: str
    reason: str


class P06ConfigurationInvalid(Exception):
    """The configuration has one or more independent issues."""

    def __init__(self, issues):
        self.issues = tuple(issues)
        if not self.issues:
            raise ValueError("at least one issue is required")
        super().__init__(f"configuration has {len(self.issues)} issue(s)")


print(P06Issue("server.port", "must be an integer"))

P06Issue(path='server.port', reason='must be an integer')


The issues are copied to a tuple so the external list cannot be mutated later to change what an existing exception reports. This is a *shallow* immutability boundary, which is sufficient because `P06Issue` is itself frozen.

### Step 2 — Validate without accidentally accepting booleans as integers

A dictionary may be missing, a value may have the wrong type, or a number may be out of range. Check structure before accessing nested keys; otherwise a `KeyError` or `AttributeError` masks the intended validation output.

In [23]:
def p06_validate(config):
    issues = []
    if not isinstance(config, dict):
        raise P06ConfigurationInvalid([P06Issue("$", "must be an object")])

    server = config.get("server")
    if not isinstance(server, dict):
        issues.append(P06Issue("server", "must be an object"))
    else:
        port = server.get("port")
        if isinstance(port, bool) or not isinstance(port, int):
            issues.append(P06Issue("server.port", "must be an integer"))
        elif not 1 <= port <= 65535:
            issues.append(P06Issue("server.port", "must be within 1..65535"))

        host = server.get("host")
        if not isinstance(host, str) or not host.strip():
            issues.append(P06Issue("server.host", "must be a non-empty string"))

    flags = config.get("features")
    if not isinstance(flags, dict):
        issues.append(P06Issue("features", "must be an object"))
    elif not isinstance(flags.get("safe_mode"), bool):
        issues.append(P06Issue("features.safe_mode", "must be a boolean"))

    if issues:
        raise P06ConfigurationInvalid(issues)
    return config

This design intentionally stops descending through a missing parent object, but continues validating independent branches. It avoids reporting misleading errors like `server.port missing` when `server` itself is not an object.

### Step 3 — See how a caller can present all issues

The exception transports structured issues; the presentation layer chooses how to display them. No exception subclass is needed for every field or every reason.

In [24]:
p06_bad = {
    "server": {"host": "", "port": True},
    "features": {"safe_mode": "yes"},
}
p06_error = capture_exception(p06_validate, p06_bad)
print(str(p06_error))
for issue in p06_error.issues:
    print(f" - {issue.path}: {issue.reason}")

configuration has 3 issue(s)
 - server.port: must be an integer
 - server.host: must be a non-empty string
 - features.safe_mode: must be a boolean


### Verify the solution

Confirm three independent issues, correct paths, rejection of malformed parents, and a valid config.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [25]:
assert isinstance(p06_error, P06ConfigurationInvalid)
assert {issue.path for issue in p06_error.issues} == {
    "server.port", "server.host", "features.safe_mode"
}
assert len(capture_exception(p06_validate, {"server": [], "features": {}}).issues) == 2
p06_good = {"server": {"host": "localhost", "port": 8080}, "features": {"safe_mode": True}}
assert p06_validate(p06_good) is p06_good
print("PASS 06 — nested multi-issue validation")

PASS 06 — nested multi-issue validation


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 07 — Suppress one expected resource error—and no others

**Scenario / challenge.** An optional configuration file may be missing. A `with` block should substitute defaults only for `FileNotFoundError`; permission errors and parse failures must still reach the caller.

**You will be able to:** write a context manager that yields exactly once; conditionally suppress an error; demonstrate non-suppression.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — What suppression actually means

A context manager’s `__exit__` can suppress an exception by returning true. With `@contextmanager`, catching an exception around `yield` and not re-raising has the same effect. This should be a narrow, explicit policy.

In [26]:
@contextmanager
def p07_optional_source(open_source):
    """Yield a file-like object, or None only when it does not exist."""
    try:
        source = open_source()
    except FileNotFoundError:
        yield None
    else:
        try:
            yield source
        finally:
            source.close()


def p07_load(open_source):
    with p07_optional_source(open_source) as source:
        if source is None:
            return {"enabled": False}
        return json.load(source)

This context manager deliberately handles a failure during acquisition. It closes only an object it successfully acquired. Errors raised inside the `with` body are not intercepted by the `FileNotFoundError` acquisition handler, and cleanup still runs.

### Step 2 — Inject file-like objects; avoid filesystem prerequisites

The fake openers make three scenarios deterministic: found, not found, and permission denied. They exercise the same error boundary without requiring particular operating-system permissions.

In [27]:
def p07_found():
    return io.StringIO('{"enabled": true}')


def p07_missing():
    raise FileNotFoundError("optional file absent")


def p07_denied():
    raise PermissionError("access denied")


print(p07_load(p07_found))
print(p07_load(p07_missing))
print(type(capture_exception(p07_load, p07_denied)).__name__)

{'enabled': True}
{'enabled': False}
PermissionError


The missing resource gets a documented fallback. A `PermissionError` is an `OSError`, but it is *not* a `FileNotFoundError`; replacing it with defaults would conceal a potentially serious deployment problem.

### Step 3 — Prove cleanup and parse failures propagate

Capture the created `StringIO` so the test can inspect its closed flag. Then feed invalid JSON: the source must close even though `json.load` raises.

In [28]:
p07_handles = []

def p07_invalid_json():
    source = io.StringIO("{bad-json}")
    p07_handles.append(source)
    return source


p07_parse_error = capture_exception(p07_load, p07_invalid_json)
print(type(p07_parse_error).__name__)
print("closed after failure:", p07_handles[-1].closed)

JSONDecodeError
closed after failure: True


### Verify the solution

Check defaulting behavior, refusal to conceal permissions, and resource closure on parser failure.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [29]:
assert p07_load(p07_missing) == {"enabled": False}
assert isinstance(capture_exception(p07_load, p07_denied), PermissionError)
assert isinstance(p07_parse_error, json.JSONDecodeError)
assert p07_handles[-1].closed
print("PASS 07 — narrowly suppressed optional resource")

PASS 07 — narrowly suppressed optional resource


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 08 — Recover from partial resource acquisition with ExitStack

**Scenario / challenge.** A report writer must acquire three resources. If opening the third resource fails, the first two must still close in reverse order; if the body fails, everything must still close.

**You will be able to:** use `ExitStack` for dynamic acquisitions and demonstrate reverse-order cleanup.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Introduce an observable resource

An object records acquisition and closure, so we can test cleanup without opening real sockets. The `close` method is idempotent here for ease of inspection.

In [30]:
class P08Resource:
    """Deterministic fake resource."""

    def __init__(self, name, events):
        self.name = name
        self.events = events
        self.closed = False
        events.append(f"open:{name}")

    def close(self):
        if not self.closed:
            self.closed = True
            self.events.append(f"close:{self.name}")


def p08_acquire(name, events):
    if name == "broken":
        raise OSError("resource acquisition failed")
    return P08Resource(name, events)

Sequential `open()` calls followed by a single cleanup block can be tricky: a partially constructed list of resources must be closed even when construction fails. `ExitStack` registers cleanup *as each acquisition succeeds*.

### Step 2 — Register each cleanup immediately

Every callback registered with `ExitStack` is called in last-in, first-out order. The `with` statement exits whether the loop finishes successfully or a later acquisition fails.

In [31]:
def p08_run(names, events, *, fail_body=False):
    with ExitStack() as stack:
        opened = []
        for name in names:
            resource = p08_acquire(name, events)
            stack.callback(resource.close)
            opened.append(resource)
        if fail_body:
            raise RuntimeError("report rendering failed")
        events.append("report:done")
        return len(opened)


p08_events = []
p08_error = capture_exception(p08_run, ["alpha", "beta", "broken"], p08_events)
print(type(p08_error).__name__)
print(p08_events)

OSError
['open:alpha', 'open:beta', 'close:beta', 'close:alpha']


The first two resources are closed despite the third failing to open, and the closure order is `beta` then `alpha`. Registration after a successful acquisition is what makes this reliable.

### Step 3 — Distinguish body failure from acquisition failure

The same stack must clean up if rendering fails *after* all resources have been acquired. No broad `except` is needed: cleanup executes and the original exception propagates.

In [32]:
p08_body_events = []
p08_body_error = capture_exception(p08_run, ["alpha", "beta"], p08_body_events, fail_body=True)
print(type(p08_body_error).__name__)
print(p08_body_events)

RuntimeError
['open:alpha', 'open:beta', 'close:beta', 'close:alpha']


### Verify the solution

Check both failure modes and successful return, including exact cleanup order.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [33]:
assert type(p08_error) is OSError
assert p08_events == ["open:alpha", "open:beta", "close:beta", "close:alpha"]
assert type(p08_body_error) is RuntimeError
assert p08_body_events[-2:] == ["close:beta", "close:alpha"]
p08_ok_events = []
assert p08_run(["a", "b"], p08_ok_events) == 2
assert p08_ok_events[-3:] == ["report:done", "close:b", "close:a"]
print("PASS 08 — partial acquisition is exception-safe")

PASS 08 — partial acquisition is exception-safe


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 09 — Avoid losing the original error when cleanup also fails

**Scenario / challenge.** A resource raises while leaving a `with` block. The body itself may already have failed. A single cleanup exception can mask the original failure unless both are deliberately reported.

**You will be able to:** model double failures; preserve both independent errors; avoid silently returning from `__exit__`.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Understand the two competing failures

If `__exit__` raises a new exception while another error is active, the cleanup error normally propagates and the body error appears only as context. A caller may need both as first-class errors. For ordinary `Exception` subclasses, `ExceptionGroup` can hold both.

In [34]:
class P09BodyFailure(Exception):
    """The work failed."""


class P09CloseFailure(Exception):
    """Closing a resource failed."""


class P09FragileResource:
    def __enter__(self):
        return self

    def __exit__(self, error_type, error, tb):
        close_error = P09CloseFailure("close failed")
        if error is not None:
            raise ExceptionGroup("body and cleanup both failed", [error, close_error]) from None
        raise close_error

This is an explicit reporting policy, not a universal rule: some applications prioritize the original error and log cleanup failures separately. Here the exercise requires that callers can inspect *both*. `ExceptionGroup` accepts `Exception` instances; the broader `BaseExceptionGroup` is needed for base exceptions such as cancellation or process-exit signals, which should not be bundled casually.

### Step 2 — Exercise a double failure

Create a body error inside the `with` block and inspect the resulting group. The two child exceptions retain their own types and messages.

In [35]:
def p09_double_failure():
    with P09FragileResource():
        raise P09BodyFailure("render failed")


p09_group = capture_exception(p09_double_failure)
print(type(p09_group).__name__)
for child in p09_group.exceptions:
    print(" -", type(child).__name__, str(child))

ExceptionGroup
 - P09BodyFailure render failed
 - P09CloseFailure close failed


A `P09CloseFailure` is not interchangeable with the body error; the caller can choose whether to retry cleanup, diagnose rendering, or fail the operation. A group preserves both facts without pretending the second error never happened.

### Step 3 — Handle the cleanup-only case independently

When the body succeeds, no grouping is necessary: the caller receives the ordinary close exception. This keeps the usual single-error path simple.

In [36]:
def p09_close_only():
    with P09FragileResource():
        return "body completed"


p09_close_error = capture_exception(p09_close_only)
print(type(p09_close_error).__name__)

P09CloseFailure


### Verify the solution

Verify the group contains both types in order and that the close-only path remains a simple exception.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [37]:
assert isinstance(p09_group, ExceptionGroup)
assert [type(child) for child in p09_group.exceptions] == [P09BodyFailure, P09CloseFailure]
assert type(p09_close_error) is P09CloseFailure
assert str(p09_group.exceptions[0]) == "render failed"
print("PASS 09 — double-failure visibility")

PASS 09 — double-failure visibility


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 10 — Place a custom failure at the right boundary of a generator

**Scenario / challenge.** A generator yields valid records until a corrupt record arrives. It must raise a typed error on the failing iteration and always perform cleanup if the caller exhausts or explicitly closes it.

**You will be able to:** understand lazy failure timing; use `finally`; test `close()` and normal exhaustion.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Remember that generator bodies run lazily

Calling a generator function creates an iterator without executing its body. The first `next()` starts processing, and later `next()` calls can fail. An exception raised in the generator propagates to the consumer on the specific iteration that triggers it.

In [38]:
class P10CorruptRecord(Exception):
    """A yielded record fails format validation."""


def p10_records(lines, events):
    events.append("started")
    try:
        for line_number, raw in enumerate(lines, start=1):
            if not raw.isdigit():
                raise P10CorruptRecord(f"invalid record at line {line_number}")
            yield int(raw)
    finally:
        events.append("closed")


p10_events = []
p10_iterator = p10_records(["5", "oops", "7"], p10_events)
print("before next:", p10_events)
print("first value:", next(p10_iterator))

before next: []
first value: 5


Only after the first `next()` does the generator append `started`. The `finally` block is not yet executed because the generator is suspended at `yield`. This explains why resource acquisition and cleanup in generators require care.

### Step 2 — Watch a later iteration fail

The second `next()` reaches the corrupt input. The generator raises the custom exception, unwinds its `finally` block, and then becomes closed.

In [39]:
p10_error = capture_exception(next, p10_iterator)
print(type(p10_error).__name__, str(p10_error))
print(p10_events)

P10CorruptRecord invalid record at line 2
['started', 'closed']


The consumer receives `P10CorruptRecord`, not an unrelated `StopIteration`. After an uncaught exception leaves the generator, a subsequent `next()` raises `StopIteration` because that generator has terminated.

### Step 3 — Close an unfinished iterator explicitly

When a caller stops reading early, `close()` injects generator termination at the suspended yield point, so `finally` is executed. This is important when a generator controls a real file or cursor.

In [40]:
p10_early_events = []
p10_early = p10_records(["1", "2"], p10_early_events)
print(next(p10_early))
p10_early.close()
print(p10_early_events)

p10_full_events = []
print(list(p10_records(["3", "4"], p10_full_events)))
print(p10_full_events)

1
['started', 'closed']
[3, 4]
['started', 'closed']


### Verify the solution

Verify failure timing, cleanup after errors, cleanup on early closure, and cleanup after exhaustion.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [41]:
assert str(p10_error) == "invalid record at line 2"
assert p10_events == ["started", "closed"]
assert p10_early_events == ["started", "closed"]
assert p10_full_events == ["started", "closed"]
assert isinstance(capture_exception(next, p10_iterator), StopIteration)
print("PASS 10 — lazy errors and generator cleanup")

PASS 10 — lazy errors and generator cleanup


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 11 — Build an exception-safe cache that never poisons entries

**Scenario / challenge.** A loader can fail transiently or report a permanent missing record. Failed loads must not be cached; successful results should be reused; only transient failures may use a previously stored stale value.

**You will be able to:** separate load/caching stages; avoid broad recovery; test cache consistency.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Create two failure classes with different policies

A missing item is a permanent domain failure; a temporary backend outage may justify using stale data. These represent different recovery actions, so callers need distinct exception types.

In [42]:
class P11CacheError(Exception):
    """Root for documented cache-loading errors."""


class P11TransientBackend(P11CacheError):
    """Backend is temporarily unavailable."""


class P11MissingItem(P11CacheError):
    """Requested item does not exist."""

Avoid the tempting `except Exception: return old_value` pattern. It would silently hide coding errors and permanent failures. A stale fallback is a specific product policy, not a catch-all error handler.

### Step 2 — Update the cache only after the loader returns

Dictionary assignment should happen after a successful load. The function below uses a fake storage dictionary so we can force each scenario deterministically.

In [43]:
def p11_get(key, loader, cache, *, refresh=False):
    if not refresh and key in cache:
        return cache[key]

    try:
        fresh = loader(key)
    except P11TransientBackend:
        if key in cache:
            return cache[key]
        raise

    cache[key] = fresh
    return fresh


p11_cache = {}
print(p11_get("A", lambda key: "v1", p11_cache))
print(p11_get("A", lambda key: "not-used", p11_cache))
print(p11_cache)

v1
v1
{'A': 'v1'}


The first lookup fills the cache. The second lookup returns the cache entry without invoking the loader. The `refresh` flag explicitly opts into contacting the backend even when data is already present.

### Step 3 — Demonstrate fallback versus propagation

With stale data, a typed transient outage returns the old value. Missing-record errors propagate. A transient outage for an uncached key also propagates because no fallback exists.

In [44]:
def p11_down(key):
    raise P11TransientBackend("backend unavailable")


def p11_missing(key):
    raise P11MissingItem("record deleted")


print("stale on outage:", p11_get("A", p11_down, p11_cache, refresh=True))
print("missing:", type(capture_exception(p11_get, "A", p11_missing, p11_cache, refresh=True)).__name__)
print("uncached outage:", type(capture_exception(p11_get, "B", p11_down, p11_cache)).__name__)

stale on outage: v1
missing: P11MissingItem
uncached outage: P11TransientBackend


### Verify the solution

Assert failures did not overwrite a valid cache entry or insert a value for the missing key.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [45]:
assert p11_cache == {"A": "v1"}
assert isinstance(capture_exception(p11_get, "A", p11_missing, p11_cache, refresh=True), P11MissingItem)
assert isinstance(capture_exception(p11_get, "B", p11_down, p11_cache), P11TransientBackend)
assert "B" not in p11_cache
assert p11_get("A", lambda key: "v2", p11_cache, refresh=True) == "v2"
assert p11_cache["A"] == "v2"
print("PASS 11 — cache state remains valid on exceptions")

PASS 11 — cache state remains valid on exceptions


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 12 — Offer strict and tolerant batch policies without changing the leaf error

**Scenario / challenge.** An ETL batch sometimes needs fail-fast behavior and sometimes needs partial results plus rejected-row details. Keep a single leaf exception, and make the chosen recovery policy explicit.

**You will be able to:** implement two public modes; preserve input position; avoid silently losing failures.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Define one row-level error with position metadata

One row has one failure reason. A tolerant caller needs the original index to correlate rejected data; a strict caller wants the first failing row as an exception.

In [46]:
class P12BadRow(Exception):
    """Row-level validation failure."""

    def __init__(self, index, reason):
        self.index = index
        self.reason = reason
        super().__init__(f"row {index}: {reason}")


def p12_convert(raw, index):
    if not isinstance(raw, str) or not raw.isdecimal():
        raise P12BadRow(index, "expected a non-negative decimal string")
    return int(raw)

The leaf exception contains *where* and *why*. Do not create a separate exception subtype for “row 3” or “row 10”; data belongs in attributes, not in a proliferation of class definitions.

### Step 2 — Make caller-selected policy visible in the API

Strict mode re-raises a failing row immediately. Tolerant mode catches only the documented row exception, preserves valid results, and returns rejected errors for inspection.

In [47]:
def p12_process_batch(rows, *, strict):
    accepted = []
    rejected = []
    for index, raw in enumerate(rows):
        try:
            accepted.append(p12_convert(raw, index))
        except P12BadRow as exc:
            if strict:
                raise
            rejected.append(exc)
    return accepted, rejected


p12_accepted, p12_rejected = p12_process_batch(["4", "bad", "9", "-2"], strict=False)
print("accepted:", p12_accepted)
print("rejected:", [(e.index, e.reason) for e in p12_rejected])

accepted: [4, 9]
rejected: [(1, 'expected a non-negative decimal string'), (3, 'expected a non-negative decimal string')]


Tolerant mode does not mean failures disappear: it returns them as observable outcomes. A caller can display them, audit them, or reject the entire batch later. Unexpected bugs still propagate because only `P12BadRow` is caught.

### Step 3 — Verify fail-fast semantics

In strict mode the first invalid row stops the operation. The exception object retains its row index for callers that want to report a precise position.

In [48]:
p12_strict_error = capture_exception(p12_process_batch, ["4", "bad", "9"], strict=True)
print(type(p12_strict_error).__name__, p12_strict_error.index, str(p12_strict_error))

P12BadRow 1 row 1: expected a non-negative decimal string


### Verify the solution

Check exact accepted results, rejected positions, and strict-mode index.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [49]:
assert p12_accepted == [4, 9]
assert [error.index for error in p12_rejected] == [1, 3]
assert type(p12_strict_error) is P12BadRow and p12_strict_error.index == 1
assert p12_process_batch(["1", "2"], strict=True) == ([1, 2], [])
print("PASS 12 — explicit strict/tolerant recovery")

PASS 12 — explicit strict/tolerant recovery


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 13 — Implement a deterministic circuit breaker with typed failures

**Scenario / challenge.** A remote operation repeatedly raises transient errors. After a threshold, the circuit must reject calls until a cooldown expires, then allow one trial. Permanent errors should not be counted as transient backend outages.

**You will be able to:** model CLOSED/OPEN/HALF_OPEN transitions, inject a clock, distinguish rejection from operation failure.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Name the different failures

A transient backend failure increments the circuit counter. A locally open circuit is a separate exception: the backend was **not even called**. That distinction matters for telemetry and caller recovery.

In [50]:
class P13Transient(Exception):
    """A backend operation failed transiently."""


class P13CircuitOpen(Exception):
    """Call refused by the local circuit breaker."""


class P13State(Enum):
    CLOSED = "closed"
    OPEN = "open"
    HALF_OPEN = "half_open"

The clock is injected rather than relying on real `time.sleep()`: tests remain fast and repeatable, and the code’s transition rules can be verified at exact time boundaries.

### Step 2 — Write the transitions before making requests

A closed circuit counts consecutive transient errors; an open circuit rejects before cooldown; after cooldown it tries a single operation. This teaching example assumes sequential calls; a concurrent production implementation would need synchronization around the half-open trial.

In [51]:
class P13CircuitBreaker:
    def __init__(self, *, threshold, cooldown, clock):
        if threshold < 1 or cooldown < 0:
            raise ValueError("threshold must be positive and cooldown non-negative")
        self.threshold = threshold
        self.cooldown = cooldown
        self.clock = clock
        self.state = P13State.CLOSED
        self.failures = 0
        self.opened_at = None

    def _open(self):
        self.state = P13State.OPEN
        self.opened_at = self.clock()

    def call(self, operation):
        if self.state is P13State.OPEN:
            if self.clock() - self.opened_at < self.cooldown:
                raise P13CircuitOpen("circuit is cooling down")
            self.state = P13State.HALF_OPEN

        try:
            result = operation()
        except P13Transient:
            if self.state is P13State.HALF_OPEN:
                self._open()
            else:
                self.failures += 1
                if self.failures >= self.threshold:
                    self._open()
            raise
        except Exception:
            # Not a classified backend outage: do not count it as transient.
            # In HALF_OPEN, reopen conservatively: the trial did not prove health.
            if self.state is P13State.HALF_OPEN:
                self._open()
            raise
        else:
            self.failures = 0
            self.opened_at = None
            self.state = P13State.CLOSED
            return result

The broad `except Exception` here does **not swallow or translate** an unknown error: it immediately re-raises after maintaining the state invariant. Its sole purpose is to avoid incorrectly marking a failed half-open probe as healthy. It does not catch `KeyboardInterrupt`, `SystemExit`, or (on modern Python) task cancellation.

### Step 3 — Walk a timeline with exact state observations

Two classified failures open the circuit. The next attempt is refused before calling the backend. Advancing a fake clock makes the next call a half-open probe; a successful probe closes the circuit.

In [52]:
p13_now = [0.0]
p13_calls = []
p13_breaker = P13CircuitBreaker(threshold=2, cooldown=10.0, clock=lambda: p13_now[0])


def p13_fail():
    p13_calls.append("backend")
    raise P13Transient("temporary outage")


def p13_succeed():
    p13_calls.append("backend")
    return "healthy"


capture_exception(p13_breaker.call, p13_fail)
print("after one:", p13_breaker.state.value)
capture_exception(p13_breaker.call, p13_fail)
print("after two:", p13_breaker.state.value)
p13_refusal = capture_exception(p13_breaker.call, p13_succeed)
print("refused:", type(p13_refusal).__name__, "backend calls:", len(p13_calls))
p13_now[0] = 10.0
print("trial:", p13_breaker.call(p13_succeed), "state:", p13_breaker.state.value)

after one: closed
after two: open
refused: P13CircuitOpen backend calls: 2
trial: healthy state: closed


There is no backend call for an open-circuit refusal. A successful trial resets the failure counter. A half-open trial that fails transiently immediately reopens the circuit at the new time, which you will test next.

### Verify the solution

Test the initial timeline and a failing half-open trial with a second isolated breaker.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [53]:
assert len(p13_calls) == 3
assert isinstance(p13_refusal, P13CircuitOpen)
assert p13_breaker.state is P13State.CLOSED and p13_breaker.failures == 0
p13_time2 = [0.0]
p13_b2 = P13CircuitBreaker(threshold=1, cooldown=3, clock=lambda: p13_time2[0])
assert isinstance(capture_exception(p13_b2.call, p13_fail), P13Transient)
p13_time2[0] = 3.0
assert isinstance(capture_exception(p13_b2.call, p13_fail), P13Transient)
assert p13_b2.state is P13State.OPEN and p13_b2.opened_at == 3.0
print("PASS 13 — explicit breaker state transitions")

PASS 13 — explicit breaker state transitions


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 14 — Classify simultaneous TaskGroup failures with except*

**Scenario / challenge.** Two independent asynchronous jobs raise different custom exceptions. An `asyncio.TaskGroup` reports them as an exception group. A caller must categorize the failures while preserving details.

**You will be able to:** understand a task group failure; use `except*` on specific types; avoid losing grouped errors.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Create task-specific error types

In a task group, one failing task cancels unfinished siblings. For a deterministic example with *two* failures, each coroutine raises synchronously when it first runs, before any suspension point; this lets the event loop collect both failures in the group.

In [54]:
class P14FetchError(Exception):
    """A fetch task failed."""


class P14ParseError(Exception):
    """A parse task failed."""


async def p14_fetch():
    raise P14FetchError("upstream rejected fetch")


async def p14_parse():
    raise P14ParseError("document structure invalid")

`TaskGroup` exits by waiting for tasks and raising an `ExceptionGroup` for non-cancellation errors. The grouped exception may contain multiple failures, which ordinary `except SpecificError` will not catch individually.

### Step 2 — Write the task group and classify with except*

Each `except*` clause receives a subgroup matching its type. We record both leaves without throwing away details. Do not mix `except` and `except*` in the same `try` statement.

In [55]:
async def p14_run():
    found = []
    try:
        async with asyncio.TaskGroup() as group:
            group.create_task(p14_fetch())
            group.create_task(p14_parse())
    except* P14FetchError as subgroup:
        found.extend(("fetch", str(error)) for error in subgroup.exceptions)
    except* P14ParseError as subgroup:
        found.extend(("parse", str(error)) for error in subgroup.exceptions)
    return found


p14_found = await p14_run()
print(sorted(p14_found))

[('fetch', 'upstream rejected fetch'), ('parse', 'document structure invalid')]


`except*` splits the group by exception type. Here both types have handlers, so the function returns the collected descriptions. In general, any unmatched subgroup automatically continues propagating after the handlers finish.

### Step 3 — Inspect the original grouping structure

A second function leaves the group unhandled. The test-only capture helper sees the top-level `ExceptionGroup` so we can inspect leaf types directly.

In [56]:
async def p14_unhandled():
    async with asyncio.TaskGroup() as group:
        group.create_task(p14_fetch())
        group.create_task(p14_parse())


p14_group = None
try:
    await p14_unhandled()
except ExceptionGroup as caught_group:
    p14_group = caught_group
    print(type(p14_group).__name__, [type(e).__name__ for e in p14_group.exceptions])

ExceptionGroup ['P14FetchError', 'P14ParseError']


### Verify the solution

Verify both error categories appear once and their messages survive classification.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [57]:
assert sorted(kind for kind, _ in p14_found) == ["fetch", "parse"]
assert any("upstream rejected fetch" in text for _, text in p14_found)
assert any("document structure invalid" in text for _, text in p14_found)
assert {type(e) for e in p14_group.exceptions} == {P14FetchError, P14ParseError}
print("PASS 14 — TaskGroup subgroup classification")

PASS 14 — TaskGroup subgroup classification


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 15 — Distinguish an async deadline from external cancellation

**Scenario / challenge.** An async service sets its own time limit on a slow dependency. A local deadline must become a custom domain timeout; cancellation initiated by another task must remain cancellation and run cleanup.

**You will be able to:** place the timeout boundary correctly and keep `CancelledError` out of translation.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Define the meaning of the custom deadline

The public service error means *this service’s own deadline elapsed*. That is different from a caller cancelling the task. Translate the `TimeoutError` raised by an `asyncio.timeout` context manager outside the context block.

In [58]:
class P15ServiceDeadline(Exception):
    """The service's own deadline expired."""


async def p15_service(delay, limit, events):
    try:
        try:
            async with asyncio.timeout(limit):
                events.append("working")
                await asyncio.sleep(delay)
                return "done"
        except TimeoutError as exc:
            raise P15ServiceDeadline("service deadline exceeded") from exc
    finally:
        events.append("cleanup")


p15_events = []
p15_deadline = None
try:
    await p15_service(0.05, 0.001, p15_events)
except P15ServiceDeadline as exc:
    p15_deadline = exc
print(type(p15_deadline).__name__, type(p15_deadline.__cause__).__name__)
print(p15_events)

P15ServiceDeadline TimeoutError
['working', 'cleanup']


`asyncio.timeout` converts its own internal cancellation into `TimeoutError` when exiting the context. External `CancelledError` is a different control-flow signal; the service does not catch or convert it. The `finally` block executes on both paths.

### Step 2 — Cancel from outside and inspect what escapes

Create a long-running task, yield once to let it start, cancel it, and catch cancellation only in the *test harness*. Do not suppress cancellation in production code unless implementing a well-defined protocol.

In [59]:
p15_cancel_events = []
p15_task = asyncio.create_task(p15_service(100, 200, p15_cancel_events))
await asyncio.sleep(0)
p15_task.cancel()
p15_was_cancelled = False
try:
    await p15_task
except asyncio.CancelledError:
    p15_was_cancelled = True
print("cancelled:", p15_was_cancelled)
print(p15_cancel_events)

cancelled: True
['working', 'cleanup']


A cancelled task should remain cancelled so parent tasks and shutdown logic can behave correctly. The cleanup event confirms that `finally` does not depend on successful completion. Tests use small times only for a local deadline; none of these examples waits for real minutes.

### Step 3 — Verify the successful path

A successful operation returns normally and still performs cleanup. A deadline error should expose its low-level timeout as `__cause__` for debugging.

In [60]:
p15_ok_events = []
print(await p15_service(0, 1, p15_ok_events))
print(p15_ok_events)

done
['working', 'cleanup']


### Verify the solution

Verify the three distinct outcomes: success, deadline exceeded, and external cancellation.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [61]:
assert isinstance(p15_deadline, P15ServiceDeadline)
assert isinstance(p15_deadline.__cause__, TimeoutError)
assert p15_events[-1] == "cleanup"
assert p15_was_cancelled and p15_cancel_events[-1] == "cleanup"
assert p15_ok_events == ["working", "cleanup"]
print("PASS 15 — deadline translation versus cancellation")

PASS 15 — deadline translation versus cancellation


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 16 — Translate SQLite integrity errors without breaking transaction rollback

**Scenario / challenge.** A tiny inventory database has a unique SKU constraint. Duplicate inserts should raise a typed domain error with the original SQLite error chained; an invalid quantity should remain a validation error; failed inserts must not leave partial writes.

**You will be able to:** use transactions, parameterized SQL, and narrow database exception translation.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Use an in-memory database with real constraints

Unlike a mock that merely raises a preselected error, SQLite will enforce an actual UNIQUE constraint. A fresh in-memory connection makes this example self-contained.

In [62]:
class P16InventoryError(Exception):
    """Base for documented inventory domain failures."""


class P16DuplicateSKU(P16InventoryError):
    """Attempted to insert an already-registered SKU."""


def p16_connection():
    connection = sqlite3.connect(":memory:")
    connection.execute("CREATE TABLE items (sku TEXT PRIMARY KEY, quantity INTEGER NOT NULL)")
    connection.commit()
    return connection


p16_db = p16_connection()
print("table ready")

table ready


Parameterized SQL (`?`) separates data from SQL syntax. We validate inputs before entering the database transaction; this example deliberately avoids turning all SQLite errors into “duplicate SKU,” which would misrepresent unrelated problems.

### Step 2 — Translate exactly the unique-constraint error

Python SQLite exposes error names such as `SQLITE_CONSTRAINT_PRIMARYKEY`. Check the vendor error code before choosing a domain interpretation; otherwise propagate the original integrity error unchanged.

In [63]:
def p16_insert(connection, sku, quantity):
    if not isinstance(sku, str) or not sku:
        raise ValueError("sku must be non-empty text")
    if isinstance(quantity, bool) or not isinstance(quantity, int) or quantity < 0:
        raise ValueError("quantity must be a non-negative integer")

    try:
        with connection:
            connection.execute("INSERT INTO items (sku, quantity) VALUES (?, ?)", (sku, quantity))
    except sqlite3.IntegrityError as exc:
        if getattr(exc, "sqlite_errorname", "") in {
            "SQLITE_CONSTRAINT_PRIMARYKEY", "SQLITE_CONSTRAINT_UNIQUE"
        }:
            raise P16DuplicateSKU(f"SKU already exists: {sku}") from exc
        raise


p16_insert(p16_db, "SKU-A", 4)
p16_duplicate = capture_exception(p16_insert, p16_db, "SKU-A", 99)
print(type(p16_duplicate).__name__)
print("cause:", type(p16_duplicate.__cause__).__name__)

P16DuplicateSKU
cause: IntegrityError


The connection context manager commits on success and rolls back on an error; it does **not** close the connection. The domain translation happens after the transaction has exited, so the failed write cannot be accidentally committed by a later operation.

### Step 3 — Inspect actual database state after failure

A duplicate write must preserve the original quantity. A validation failure must not add any row. We query the database rather than trusting the exception alone.

In [64]:
p16_rows = p16_db.execute("SELECT sku, quantity FROM items ORDER BY sku").fetchall()
print(p16_rows)
p16_bad_quantity = capture_exception(p16_insert, p16_db, "SKU-B", -1)
print(type(p16_bad_quantity).__name__)

[('SKU-A', 4)]
ValueError


### Verify the solution

Confirm exception type, chain, successful transaction, duplicate rollback, and validation isolation.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [65]:
assert type(p16_duplicate) is P16DuplicateSKU
assert isinstance(p16_duplicate.__cause__, sqlite3.IntegrityError)
assert p16_rows == [("SKU-A", 4)]
assert type(p16_bad_quantity) is ValueError
assert p16_db.execute("SELECT COUNT(*) FROM items").fetchone()[0] == 1
p16_insert(p16_db, "SKU-B", 2)
assert p16_db.execute("SELECT COUNT(*) FROM items").fetchone()[0] == 2
p16_db.close()
print("PASS 16 — real constraint and rollback")

PASS 16 — real constraint and rollback


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 17 — Combine exception behavior with an ordinary mixin safely

**Scenario / challenge.** The source tutorial introduces inheriting from multiple exception bases. You want a diagnostic formatting helper without coupling two different built-in exception memory layouts.

**You will be able to:** separate reusable behavior from exception taxonomy; inspect MRO; avoid fragile multiple built-in exception inheritance.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Understand the design trade-off

Python allows multiple inheritance, but the official documentation cautions against subclassing multiple built-in exception types because their internal layouts can conflict. Here a normal mixin adds behavior, while one `Exception`-derived root controls catchability.

In [66]:
class P17DiagnosticMixin:
    """Regular Python mixin; not another exception base."""

    def diagnostic_record(self):
        return {
            "type": type(self).__name__,
            "code": self.code,
            "message": str(self),
        }


class P17PipelineError(P17DiagnosticMixin, Exception):
    """One exception root with reusable formatting behavior."""

    code = "PIPELINE_ERROR"


class P17SchemaMismatch(P17PipelineError):
    """Input schema differs from the expected contract."""

    code = "SCHEMA_MISMATCH"

A plain mixin provides methods, not another independent exception-catching identity. This avoids the initialization and memory-layout ambiguity that can arise with classes inheriting from multiple built-in exception types.

### Step 2 — Inspect actual method lookup and catchability

Python’s method resolution order (MRO) shows that `P17DiagnosticMixin` contributes methods and `Exception` contributes familiar exception behavior. The leaf stays catchable by the library root.

In [67]:
p17_error = P17SchemaMismatch("missing column: account_id")
print([cls.__name__ for cls in P17SchemaMismatch.__mro__])
print(p17_error.diagnostic_record())
print(isinstance(p17_error, P17PipelineError), isinstance(p17_error, Exception))

['P17SchemaMismatch', 'P17PipelineError', 'P17DiagnosticMixin', 'Exception', 'BaseException', 'object']
{'type': 'P17SchemaMismatch', 'code': 'SCHEMA_MISMATCH', 'message': 'missing column: account_id'}
True True


This example is consistent with the supplied tutorial’s broader goal: create errors that can be handled by their specific class or by an application-level base. We keep reusable formatting orthogonal to the catch hierarchy.

### Step 3 — Keep presentation free from executable code or secret data

The record is an ordinary dictionary, ready for controlled formatting. For a real application, explicitly filter any fields before logging or serializing it; this example’s message is intentionally nonsensitive.

In [68]:
p17_text = json.dumps(p17_error.diagnostic_record(), sort_keys=True)
print(p17_text)

{"code": "SCHEMA_MISMATCH", "message": "missing column: account_id", "type": "P17SchemaMismatch"}


### Verify the solution

Check exception catchability, inherited mixin method, and base Exception args.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [69]:
assert isinstance(p17_error, P17PipelineError)
assert p17_error.args == ("missing column: account_id",)
assert p17_error.diagnostic_record()["code"] == "SCHEMA_MISMATCH"
assert "P17DiagnosticMixin" in [cls.__name__ for cls in P17SchemaMismatch.__mro__]
print("PASS 17 — regular mixin plus single exception root")

PASS 17 — regular mixin plus single exception root


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

## Problem 18 — CAPSTONE — Build a staged import pipeline with typed, recoverable failures

**Scenario / challenge.** Build an in-memory importer that decodes bytes, parses JSON, validates a record, transforms it, and writes into a transactional store. Error classes must identify the failing stage; successful writes must be atomic; audit events must mark failures without accidentally leaking raw input.

**You will be able to:** integrate exception hierarchy, narrow translation with causes, structured metadata, atomic updates, and end-to-end tests.

Work through the small steps **in order**. Read the explanation after each example before running the next one. The complete solution is assembled incrementally; the final cell is an executable check, not just a displayed answer.

### Step 1 — Write the explicit error contract before the implementation

Separate decode, parsing, validation, and duplicate-record failures. A root error lets the application boundary catch all documented import failures without hiding unrelated programmer bugs. Each leaf carries a safe, fixed code.

In [70]:
class P18ImportError(Exception):
    """Base class for importer-owned failures."""

    code = "IMPORT_ERROR"
    safe_message = "Import failed."

    def __init__(self, detail, *, record_id=None):
        self.record_id = record_id
        super().__init__(detail)


class P18DecodeError(P18ImportError):
    code = "DECODE_ERROR"
    safe_message = "File encoding is invalid."


class P18ParseError(P18ImportError):
    code = "PARSE_ERROR"
    safe_message = "File content is not valid JSON."


class P18ValidationError(P18ImportError):
    code = "VALIDATION_ERROR"
    safe_message = "Required fields are invalid."


class P18DuplicateRecord(P18ImportError):
    code = "DUPLICATE_RECORD"
    safe_message = "The record already exists."

The internal `detail` is diagnostic. A public API should send `safe_message` and `code`, not stringify the internal exception or its cause. `record_id` is optional because decoding and parsing can fail before an ID can be extracted.

### Step 2 — Translate decoding and parsing errors at their own stages

Each function catches only the low-level exceptions expected from *its* operation. The original exception is explicitly chained for debugging. A JSON top-level value need not be an object, so shape validation is the next stage.

In [71]:
def p18_decode(payload):
    if not isinstance(payload, bytes):
        raise TypeError("payload must be bytes")
    try:
        return payload.decode("utf-8")
    except UnicodeDecodeError as exc:
        raise P18DecodeError("input is not UTF-8") from exc


def p18_parse(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise P18ParseError(f"invalid JSON at line {exc.lineno}") from exc


print(p18_parse(p18_decode(b'{"id": "r-1", "amount": 2}')))

{'id': 'r-1', 'amount': 2}


Notice a different policy for wrong **programmer input**: if the caller passes a `str` instead of `bytes`, `TypeError` passes through. A public importer should not misleadingly report “UTF-8 encoding invalid” for an incorrect Python argument type.

### Step 3 — Validate the complete record before touching the store

Shape and business rules live in one pure function. Reject booleans for amount, require a non-empty string identifier, and do not coerce silently. Only a valid record can enter the write stage.

In [72]:
def p18_validate(document):
    if not isinstance(document, dict):
        raise P18ValidationError("top-level JSON value must be an object")
    record_id = document.get("id")
    amount = document.get("amount")
    if not isinstance(record_id, str) or not record_id.strip():
        raise P18ValidationError("id must be non-empty text")
    if isinstance(amount, bool) or not isinstance(amount, int) or amount < 0:
        raise P18ValidationError("amount must be a non-negative integer", record_id=record_id)
    return {"id": record_id, "amount_cents": amount * 100}


print(p18_validate({"id": "r-1", "amount": 2}))

{'id': 'r-1', 'amount_cents': 200}


The transformation produces a **new** dictionary and avoids mutating the input document. This makes the validation stage testable in isolation and prevents partial store updates when a field is invalid.

### Step 4 — Implement an atomic write and keep an independent audit trail

The store uses copy-on-write: build a candidate state, enforce the duplicate constraint, then swap the candidate into place. Audit events are appended outside the store transaction because failures must be observable even when a write is rolled back. They contain only a safe code, never payload bytes.

In [73]:
class P18Store:
    def __init__(self):
        self.records = {}
        self.audit = []

    def write(self, record):
        candidate = dict(self.records)
        record_id = record["id"]
        if record_id in candidate:
            raise P18DuplicateRecord("duplicate record", record_id=record_id)
        candidate[record_id] = dict(record)
        self.records = candidate


def p18_import(payload, store):
    try:
        text = p18_decode(payload)
        document = p18_parse(text)
        record = p18_validate(document)
        store.write(record)
    except P18ImportError as exc:
        store.audit.append({"outcome": "failed", "code": exc.code})
        raise
    else:
        store.audit.append({"outcome": "accepted", "code": "OK"})
        return record


p18_store = P18Store()
print(p18_import(b'{"id": "r-1", "amount": 2}', p18_store))
print(p18_store.records, p18_store.audit)

{'id': 'r-1', 'amount_cents': 200}
{'r-1': {'id': 'r-1', 'amount_cents': 200}} [{'outcome': 'accepted', 'code': 'OK'}]


All documented stage failures share `P18ImportError`, so one outer handler can safely record a fixed code. Unexpected errors still propagate: pretending an arbitrary `KeyError` or `TypeError` is a bad customer record would make bugs harder to discover. A real multi-process database would need a real database transaction or concurrency control; this in-memory copy-on-write technique is not a substitute for one.

### Step 5 — Write an application-boundary response without exposing internal data

The final boundary chooses a stable public dictionary for expected import errors. It never serializes `str(exc)` or `exc.__cause__`. For this demonstration, the error object is returned separately for diagnostic tests; production APIs should not return it to users.

In [74]:
def p18_public_result(payload, store):
    try:
        result = p18_import(payload, store)
    except P18ImportError as exc:
        return {"ok": False, "error": {"code": exc.code, "message": exc.safe_message}}
    else:
        return {"ok": True, "record": result}


p18_payloads = [
    b"\xff",                                # invalid UTF-8 bytes
    b'{"id":',                            # invalid JSON
    b'{"id": "r-2", "amount": true}',   # valid JSON; invalid domain amount
    b'{"id": "r-1", "amount": 9}',      # duplicate record
    b'{"id": "r-2", "amount": 3}',      # successful new record
]
for payload in p18_payloads:
    print(p18_public_result(payload, p18_store))

{'ok': False, 'error': {'code': 'DECODE_ERROR', 'message': 'File encoding is invalid.'}}
{'ok': False, 'error': {'code': 'PARSE_ERROR', 'message': 'File content is not valid JSON.'}}
{'ok': False, 'error': {'code': 'VALIDATION_ERROR', 'message': 'Required fields are invalid.'}}
{'ok': False, 'error': {'code': 'DUPLICATE_RECORD', 'message': 'The record already exists.'}}
{'ok': True, 'record': {'id': 'r-2', 'amount_cents': 300}}


The public result uses a fixed code and fixed message; it does not print raw records on failures. The failed attempts must not alter `records`. The success path creates exactly one new row. This is an end-to-end test of the taxonomy, not merely a unit test of one class.

### Step 6 — Inspect the error cause and the final state separately

A decode failure is chained from `UnicodeDecodeError`. A duplicate failure has no lower-level cause because it is identified directly by the store. Tests must distinguish a meaningful cause from a manufactured one.

In [75]:
p18_decode_failure = capture_exception(p18_import, b"\xff", P18Store())
p18_duplicate_failure = capture_exception(
    p18_import, b'{"id": "r-1", "amount": 50}', p18_store
)
print(type(p18_decode_failure).__name__, type(p18_decode_failure.__cause__).__name__)
print(type(p18_duplicate_failure).__name__, p18_duplicate_failure.record_id)
print("committed ids:", sorted(p18_store.records))

P18DecodeError UnicodeDecodeError
P18DuplicateRecord r-1
committed ids: ['r-1', 'r-2']


### Verify the solution

Exercise all error categories, check that failed imports do not commit, and ensure the public contract contains no raw bytes or internal messages.

The `assert` statements are executable specifications. If they do not raise, the observed behavior matches the required contract.

In [76]:
assert isinstance(p18_decode_failure, P18DecodeError)
assert isinstance(p18_decode_failure.__cause__, UnicodeDecodeError)
assert type(p18_duplicate_failure) is P18DuplicateRecord
assert p18_duplicate_failure.record_id == "r-1"
assert sorted(p18_store.records) == ["r-1", "r-2"]
assert p18_store.records["r-1"]["amount_cents"] == 200
assert p18_store.records["r-2"]["amount_cents"] == 300
assert len(p18_store.audit) == 7  # initial success + 5 cases + 1 diagnostic call to this store
assert [item["outcome"] for item in p18_store.audit].count("accepted") == 2
assert all(set(item) == {"outcome", "code"} for item in p18_store.audit)
p18_bad_response = p18_public_result(b"\xff", P18Store())
assert p18_bad_response == {
    "ok": False,
    "error": {"code": "DECODE_ERROR", "message": "File encoding is invalid."},
}
assert "\xff" not in json.dumps(p18_bad_response)
assert isinstance(capture_exception(p18_import, "not bytes", P18Store()), TypeError)
print("PASS 18 — integrated staged importer and public error boundary")

PASS 18 — integrated staged importer and public error boundary


**Why this matters.** A custom exception is part of a function’s public interface. Tests should verify its *type, attributes, cause, and recovery behavior* rather than merely that some exception occurred.

# Further reasoning exercises (with worked answers)

These questions focus on design reasoning. The explanations below are solutions, not additional library code.

**A. Why not define `class AppError(BaseException)`?** Most application-level failures should inherit from `Exception`. `BaseException` also encompasses control-flow exceptions such as `KeyboardInterrupt` and `SystemExit`, which normal application recovery should not capture accidentally.

**B. Why does `raise DomainError(...) from None` not guarantee secrecy?** It suppresses display of implicit context; the underlying exception may still exist as `__context__`, and independent logs may contain sensitive text. Never put secrets into exceptions in the first place, and use an explicit safe presentation contract.

**C. Can a context manager always swallow `FileNotFoundError`?** No. It depends on the semantic boundary: absence of an optional configuration is recoverable, but absence of a required financial ledger is an error. Recovery rules come from the application contract, not from exception names alone.

**D. Should you use multiple inheritance such as `class MyError(ValueError, OSError)`?** Avoid it: built-in exceptions may have incompatible internal implementations and differing initialization contracts. If compatibility with an existing built-in error is necessary, choose an intentional single inheritance chain or a separate, documented adapter; ordinary behavior mixins can be composed safely.

**E. What should an API log and expose?** Logs may retain appropriately access-controlled internal diagnostic details (without secrets). Public responses should expose fixed safe messages and stable error codes, never indiscriminately stringify arbitrary exception objects.

**F. When should you use `ExceptionGroup`?** When multiple independent errors truly need to reach the caller as failures. For ordinary tolerant processing, returning accepted results plus a list of rejected rows may communicate the intended API contract more naturally.

# Final checklist

- [ ] Choose an `Exception`-derived application root only when grouping documented failures helps callers.
- [ ] Keep common Python `TypeError` and `ValueError` behaviors meaningful where appropriate.
- [ ] Define focused subclasses when a caller can perform a different recovery action.
- [ ] Keep structured fields validated and `Exception.args` coherent.
- [ ] Prefer `raise DomainError(...) from exc` for deliberate translation; understand how context works.
- [ ] Add diagnostic notes without including secrets, and use bare `raise` to preserve the original error.
- [ ] Catch only recoverable exceptions; preserve cancellation, unexpected bugs, and cleanup semantics.
- [ ] Protect resource acquisition, transaction state, and caches against partial failure.
- [ ] Test observable exception behavior and state after an error, not only a printed message.

**You have finished the tutorial when every problem’s final cell prints `PASS` in a fresh kernel.**